# Week 2 — Models, deployments, prompting, and the first response

Use benchmarks to shortlist models, then compare them on task-specific cases. Record deployment type, region, version, quota, context limits, latency, quality, and cost before choosing a default.

In [ ]:
import importlib.util
import sys
from pathlib import Path

curriculum_root = next(
    candidate
    for base in (Path.cwd(), *Path.cwd().parents)
    for candidate in (base, base / "examples" / "foundry-curriculum")
    if (candidate / "notebook_setup.py").is_file()
)
spec = importlib.util.spec_from_file_location(
    "foundry_curriculum_setup", curriculum_root / "notebook_setup.py"
)
helpers = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = helpers
spec.loader.exec_module(helpers)
session = helpers.load_session(curriculum_root)
labs = helpers.load_offline_labs(curriculum_root)
session.safe_summary()

In [ ]:
model_cards = {
    "candidate-standard": {
        "deployment_type": "GlobalStandard",
        "region": "verify-before-connected-use",
        "model_version": "immutable-fixture-a",
        "quota_checked": False,
        "context_limit_checked": False,
    },
    "candidate-efficient": {
        "deployment_type": "GlobalStandard",
        "region": "verify-before-connected-use",
        "model_version": "immutable-fixture-b",
        "quota_checked": False,
        "context_limit_checked": False,
    },
}
model_cards

In [ ]:
selection_thresholds = labs.ModelThresholds(
    minimum_quality=0.90,
    maximum_p95_latency_ms=500,
    maximum_average_cost_units=0.90,
    minimum_cases=3,
)
model_cases = (
    labs.ModelCase("endpoint", ("project endpoint", "deployment")),
    labs.ModelCase("identity", ("Entra", "RBAC")),
    labs.ModelCase("structured", ("validate", "untrusted")),
)
selection_thresholds

In [ ]:
def observation(output, latency_ms, cost_units):
    return labs.ModelObservation(output, latency_ms, cost_units)


offline_observations = {
    "candidate-standard": {
        "endpoint": observation("Use a project endpoint and deployment.", 420, 0.80),
        "identity": observation("Entra authenticates; RBAC authorizes.", 480, 0.82),
        "structured": observation("Validate every untrusted output.", 460, 0.81),
    },
    "candidate-efficient": {
        "endpoint": observation("Use a project endpoint and deployment.", 250, 0.40),
        "identity": observation("Entra authenticates; RBAC authorizes.", 280, 0.42),
        "structured": observation("Validate the response shape.", 270, 0.41),
    },
}
offline_provider = labs.FakeTextResponseBoundary(offline_observations)

In [ ]:
deployments = tuple(model_cards)
model_results = labs.compare_models(offline_provider, deployments, model_cases)
model_decision = labs.select_model(model_results, selection_thresholds)
small_results = labs.compare_models(offline_provider, deployments, model_cases[:1])
underpowered_decision = labs.select_model(small_results, selection_thresholds)
assert {result.case_ids for result in model_results} == {
    tuple(case.case_id for case in model_cases)
}
assert model_decision == labs.ModelDecision(
    "adopt", "candidate-standard", "all thresholds passed"
)
assert underpowered_decision.decision == "inconclusive"
{"results": model_results, "decision": model_decision}

## Optional connected call

Review the prompt for sensitive data and confirm the deployment, role, quota, and expected cost. The default path makes no network request.

In [ ]:
RUN_CONNECTED = False
prompt = (
    "In three concise bullets, explain the difference between a Foundry "
    "project endpoint and a model deployment name."
)

if RUN_CONNECTED:
    response = helpers.create_text_response(session, prompt, allow_network=True)
    print(response.output_text)
else:
    print("Connected call skipped. Configuration was validated only.")

## Exit criteria

Compare at least two deployments on the same versioned cases. Declare the quality, latency, and cost thresholds before reviewing results; record an inconclusive decision when evidence is insufficient.